This is a script that Scraps the House price Data from Zoopla.co.uk , 
The Records collected include 
- Price sold
- Address
- Number of rooms
- Number of Baths
- Number of toilet
- EPC Rating
- Square foot meter SQM
- Actual price
- Minimum Price
- Maximum Price
- Longitude
- Latitude
- UPRN
- Post code
- post town name

## How the Script works

**step 1**
Make Sure that the *Libararies* below used for analysis of this project are installed on your jupyter notebook.

**step 2**
The script scrapes data based on the outcode of a post code e.g LS1 1AA Leeds is the post code, Just enter "LS1" on the **search** below, you can enter as much outcodes you want to scrape. note that the BASE_URL is for scrapping *rental prices*

**step 3**
Once **step 2** is completed then you can go to "# Main script execution" comment section to increase or decrease the number of pages that you want to be scrapped from zoopla. by default the available number of pages is *40pages* 

**step 4**
When you run the script, you will noticed that you Google chrome will open Zoopla website, and the sript will web crawl through the listings while it is collecting the data needed for analysis

**step 5**
    once the **step 4** has been done, the data is now stored in a csv file

In [5]:
# libraries needed for webscrapping
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from undetected_chromedriver import Chrome
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.remote.webdriver import WebDriver
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.common.action_chains import ActionChains
from typing import cast
from collections.abc import Iterator
import pandas as pd
from bs4 import BeautifulSoup
import re

# Constants

# outcodes to search 
search = ['HD1','HD2','HD3','HD4','HD5','HD6','HD7','HD8,'HD9']


# zoopla url
URL = "https://www.zoopla.co.uk/house-prices/{outcode}/?new_homes=include&q={outcode}&view_type=list&pn="
TIMEOUT = 5

# Helper function to extract text from a WebElement
def etext(e: WebElement) -> str:
    if e:
        if t := e.text.strip():
            return t
        if (p := e.get_property("textContent")) and isinstance(p, str):
            return p.strip()
    return ""

# Click the WebElement
def click(driver: WebDriver, e: WebElement) -> None:
    ActionChains(driver).click(e).perform()

# Get all WebElements that match the given CSS selector
def get_all(driver: WebDriver, css: str) -> Iterator[WebElement]:
    wait = WebDriverWait(driver, TIMEOUT)
    sel = (By.CSS_SELECTOR, css)
    try:
        yield from wait.until(EC.presence_of_all_elements_located(sel))
    except TimeoutException:
        pass

# Look for the Next button and click it
def click_next(driver: WebDriver) -> None:
    for a in get_all(driver, "a[aria-live=polite] > div > div:nth-child(2)"):
        if etext(a) == "Next":
            click(driver, a)
            break


# Handle cookie consent popup
def click_through(driver: WebDriver) -> None:
    try:
        wait = WebDriverWait(driver, TIMEOUT)
        shadow_root = driver.find_element(By.ID, "usercentrics-root").shadow_root
        button = wait.until(EC.element_to_be_clickable(
            (By.CSS_SELECTOR, "button[data-testid=uc-deny-all-button]")
        ))
        click(driver, button)
    except Exception:
        pass  # Ignore if cookies popup is not present


# Get additional details (EPC rating, stations, schools)
def get_details(driver, listing_url):
    driver2 = Chrome()
    driver2.get(listing_url)
    click_through(driver2)  # Handle cookies

    try:
        # EPC Rating
        epc_element = WebDriverWait(driver2, 5).until(
            EC.presence_of_element_located((By.XPATH, ".//*[contains(text(), 'EPC rating') or contains(text(), 'EPC Rating')]"))
        )
        epc_rating = etext(epc_element)
    except :
        epc_rating = "N/A"

    try:
        soup = BeautifulSoup(driver2.page_source, "html.parser")
        mapUrl = soup.find('source')['srcset']
        match = re.search(r"/(-?\d+\.\d+),(-?\d+\.\d+),", mapUrl)
        
        latitude = match.group(1)
        longitude = match.group(2)
        
    except:
        latitude = 'N/A'
        longitude = 'N/A'

    try:
        uprn = WebDriverWait(driver2, 5).until(EC.presence_of_element_located((By.XPATH, ".//*[contains(text(), 'uprn')]"))
        )
        uprn =etext(uprn)
        match = re.search(r'"uprn"\s*:\s*"(\d+)"', uprn)
        if match:
            uprn = match.group(1)
        else:
            uprn = 'N/A'
    except:
        uprn = 'N/A'


    try:
        saleEstimate = WebDriverWait(driver2, 5).until(EC.presence_of_element_located((By.XPATH, ".//*[contains(text(), 'saleEstimate')]"))
        )
        saleEstimate =etext(saleEstimate)
        match = re.search(r'"saleEstimate"\s*:\s*{[^}]*?"lowerPrice"\s*:\s*(\d+),\s*"currentPrice"\s*:\s*(\d+),\s*"upperPrice"\s*:\s*(\d+)', saleEstimate)
        if match:
            lowerPrice = int(match.group(1))
            currentPrice = int(match.group(2))
            upperPrice = int(match.group(3))
        else:
            lowerPrice = 'N/A'
            currentPrice = 'N/A'
            upperPrice = 'N/A'
    except:
            lowerPrice = 'N/A'
            currentPrice = 'N/A'
            upperPrice = 'N/A'

    try:
        postTownName = WebDriverWait(driver2, 5).until(EC.presence_of_element_located((By.XPATH, ".//*[contains(text(), 'postTownName')]"))
        )
        postTownName =etext(postTownName)
        match = re.search(r'"postTownName"\\?":\\?"([^"]+)"', postTownName)
        if match:
            postTownName = match.group(1)
        else:
            postTownName = 'N/A'
    except:
        postTownName = 'N/A'

            
    driver2.quit()
    return epc_rating, longitude ,latitude,uprn,lowerPrice,currentPrice,upperPrice,postTownName

    

# Scrape data from each page
def scrape_page(driver: WebDriver) -> list[dict]:
    result = []
    for house in get_all(driver, "div[data-testid=result-item]"):
        try:
            listing_url = house.find_element(By.XPATH, ".//a[starts-with(@href, '/property/')]").get_attribute("href")
            epc_rating,longitude,latitude,uprn,lowerPrice,currentPrice,upperPrice,postTownName = get_details(driver, listing_url)
            # , stations, schools, latitude,longitude = get_details(driver, listing_url)
            address = etext(house.find_element(By.CSS_SELECTOR, "h2"))
            Latestdate_sold = etext(house.find_element(By.CSS_SELECTOR, "div._1i39aq44 > div:nth-child(1) > ul > li:nth-child(1)"))
            try:
                Previousdate_sold_1 = etext(house.find_element(By.CSS_SELECTOR, "div._1i39aq44 > div:nth-child(1) > ul > li:nth-child(2)"))
            except:
                Previousdate_sold_1 = " "
            # Handle missing or empty Previousdate_sold_2 field
            try:
                Previousdate_sold_2 = etext(house.find_element(By.CSS_SELECTOR, "div._1i39aq44 > div:nth-child(1) > ul > li:nth-child(3)"))
            except:
                Previousdate_sold_2 = " "  # Set to None if missing or blank
            try:
                House_Type = etext(house.find_element(By.CSS_SELECTOR, "div._1i39aq44 > div:nth-child(1) > div > div > div._1pbf8i51 > div._1pbf8i52"))
            except:
                House_Type = " "
            try:
                Number_of_rooms = etext(house.find_element(By.CSS_SELECTOR, "div._1i39aq44 > div:nth-child(1) > div > div > div._1pbf8i51 > div:nth-child(2)"))
            except:
                Number_of_rooms = " "
            try: 
                Number_of_bath = etext(house.find_element(By.CSS_SELECTOR, "div._1i39aq44 > div:nth-child(1) > div > div > div._1pbf8i51 > div:nth-child(3)"))
            except:
                Number_of_bath = " "
            try:
                Reception = etext(house.find_element(By.CSS_SELECTOR, "div._1i39aq44 > div:nth-child(1) > div > div > div._1pbf8i51 > div:nth-child(4)"))
            except:
                Reception = " "
            try:
                Tenure = etext(house.find_element(By.CSS_SELECTOR, "div._1i39aq44 > div:nth-child(1) > div > div > div.agepcz0 > div:nth-child(1) > div"))
            except:
                Tenure = ""
            try:
                Square_foot = etext(house.find_element(By.CSS_SELECTOR, "div._1i39aq44 > div:nth-child(1) > div > div > div.agepcz0 > div:nth-child(2) > div"))
            except:
                Square_foot = " "
            result.append({"Address": address,"Date Last Sold": Latestdate_sold,"Previous date sold(-1)": Previousdate_sold_1,
                         "Previous date sold(-2)":Previousdate_sold_2,"Property Type": House_Type,"Number of rooms": Number_of_rooms,
                          "Number of Bath": Number_of_bath,"Reception" : Reception,"Tenure": Tenure , "Square foot" : Square_foot,
                          "EPC Rating": epc_rating,"UPRN":uprn,"lowerPrice":lowerPrice,"currentPrice":currentPrice,
                           "upperPrice":upperPrice,"longitude": longitude,"latitude": latitude,"Listing URL": listing_url})

        except NoSuchElementException:
            continue  # Skip missing elements
    return result


# Main script execution
if __name__ == "__main__":
    all_results = []
    max_pages = 40

    with Chrome() as driver:
        for outcode in search:
            print(f"🔍 Searching: {outcode}")
            for page in range(1, max_pages + 1):
                url = URL.format(outcode=outcode, page=page) + str(page)
                print(f"Scraping page {page} of {outcode} → {url}")
                driver.get(url)
                click_through(driver)

                page_results = scrape_page(driver)
                if not page_results:
                    print(f"No results on page {page}, stopping early for {outcode}")
                    break  # Optional: stop if no results found on a page

                all_results.extend(page_results)

    # Convert results to DataFrame
    df = pd.DataFrame(all_results)
    print(df)
    print(f"✅ Scraping complete. Total listings scraped: {len(df)}")

🔍 Searching: HD9
Scraping page 1 of HD9 → https://www.zoopla.co.uk/house-prices/HD9/?new_homes=include&q=HD9&view_type=list&pn=1
Scraping page 2 of HD9 → https://www.zoopla.co.uk/house-prices/HD9/?new_homes=include&q=HD9&view_type=list&pn=2
Scraping page 3 of HD9 → https://www.zoopla.co.uk/house-prices/HD9/?new_homes=include&q=HD9&view_type=list&pn=3
Scraping page 4 of HD9 → https://www.zoopla.co.uk/house-prices/HD9/?new_homes=include&q=HD9&view_type=list&pn=4
Scraping page 5 of HD9 → https://www.zoopla.co.uk/house-prices/HD9/?new_homes=include&q=HD9&view_type=list&pn=5
Scraping page 6 of HD9 → https://www.zoopla.co.uk/house-prices/HD9/?new_homes=include&q=HD9&view_type=list&pn=6
Scraping page 7 of HD9 → https://www.zoopla.co.uk/house-prices/HD9/?new_homes=include&q=HD9&view_type=list&pn=7
Scraping page 8 of HD9 → https://www.zoopla.co.uk/house-prices/HD9/?new_homes=include&q=HD9&view_type=list&pn=8
Scraping page 9 of HD9 → https://www.zoopla.co.uk/house-prices/HD9/?new_homes=include&q

In [7]:
 # Save to CSV
df.to_csv(r"C:\Users\obinn\OneDrive\Desktop\Data\House\Kirklees\HD9 (1-40).csv", index=False)

# Merge the different csv file

In [1]:
import pandas as pd

In [14]:
# list the csv files to merge 
csv_files= [r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO1 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO2,21 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO4,41,42 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO7 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO8 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO10,Y11 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO12,Y13 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO14,Y15 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO16 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO17 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO18,19 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO22,23,24 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO25,26,3 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO30,31,32 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO43,5 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO51,6 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO60 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO61 (1-40).csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\YO62 (1-40).csv"
           ]

df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# Save to a new CSV 
df.to_csv(r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\York.csv", index=False)

In [15]:
# list the csv files to merge 
csv_files= [r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\Bradford\Bradford.csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\Calderdale\Calderdale.csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\Doncaster\Doncaster.csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\Harrogate\Harrogate.csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\Hull\Hull.csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\Kirklees\Kirklees.csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\Leeds\Leeds.csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\Shelfield\Shelfield.csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\Wakefield\Wakefield.csv",
            r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\York\York.csv"
            
           ]

df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# Save to a new CSV 
df.to_csv(r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\House price\1_raw_data\All regions\Yorkshire and the Humber House prices.csv", index=False)

In [2]:
import pandas as pd

In [6]:
df=pd.read_csv(r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\1_raw_data\1.1_house_price\3_outputs\0.1_raw_house_city_master.csv")
df

,Address,Date Last Sold,Previous date sold(-1),Previous date sold(-2),Property Type,Number of rooms,Number of Bath,Reception,Tenure,Square foot,EPC Rating,UPRN,lowerPrice,currentPrice,upperPrice,longitude,latitude,Listing URL
0,"Sunbridge Halls, 178 Sunbridge Road, Bradford,...","Last sold Jan 2025\n£13,500","Last sold\nOct 2022\n£10,000","Last sold\nSept 2022\n£10,500",Flat/Maisonette,1,1,1,Leasehold,,NaN,1.000519e+11,67000.0,84000.0,100000.0,53.796735,-1.764231,https://www.zoopla.co.uk/property/uprn/1000519...
1,"Flat 1, Sunbridge House, 80 Kirkgate, Bradford...","Last sold Dec 2024\n£55,000","Last sold\nDec 2006\n£148,400",,Flat/Maisonette,3,2,1,Leasehold,76 sqm,EPC rating: D,1.007006e+10,69000.0,87000.0,104000.0,53.794068,-1.756230,https://www.zoopla.co.uk/property/uprn/1007005...
2,"Flat 26, Equity Chambers, 40 Piccadilly, Bradf...","Last sold Dec 2024\n£65,000","Last sold\nSept 2006\n£142,000",,Converted flat,2,2,1,Leasehold,89 sqm,EPC rating: D,1.001058e+10,78000.0,97000.0,117000.0,53.796419,-1.754395,https://www.zoopla.co.uk/property/uprn/1001058...
3,"Apartment 42, City Exchange, 61 Hall Ings, Bra...","Last sold Dec 2024\n£159,000",,,,,,,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.zoopla.co.uk/property/uprn/1009486...
4,"Apartment 109, City Exchange, 61 Hall Ings, Br...","Last sold Nov 2024\n£182,000","Last sold\nDec 2018\n£75,000",,,,,,NaN,,NaN,1.009487e+10,59000.0,74000.0,89000.0,53.791212,-1.752619,https://www.zoopla.co.uk/property/uprn/1009486...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177839,"23 Garth Drive, Hambleton, Selby, YO8 9QD","Last sold Oct 2023\n£258,000","Last sold\nMay 2021\n£233,000","Last sold\nNov 2017\n£185,000",Detached house,3,1,1,Freehold,85 sqm,EPC rating: D,1.000505e+11,276000.0,291000.0,305000.0,53.767160,-1.169669,https://www.zoopla.co.uk/property/uprn/1000505...
177840,"44 Leeds Road, Selby, YO8 4HX","Last sold Oct 2023\n£220,000","Last sold\nFeb 2006\n£142,500","Last sold\nOct 2003\n£125,000",Mid terrace house,3,1,2,Freehold,95 sqm,EPC rating: D,1.000505e+11,233000.0,245000.0,258000.0,53.780328,-1.084440,https://www.zoopla.co.uk/property/uprn/1000505...
177841,"116 Main Road, Drax, Selby, YO8 8NT","Last sold Oct 2023\n£130,000","Last sold\nApr 2019\n£86,000","Last sold\nNov 2006\n£105,000",End terrace house,2,1,2,Freehold,102 sqm,NaN,1.000505e+11,141000.0,148000.0,155000.0,53.728995,-0.978602,https://www.zoopla.co.uk/property/uprn/1000505...
177842,"9 St Marys Court, Hambleton, Selby, YO8 9GN","Last sold Oct 2023\n£190,000","Last sold\nFeb 2015\n£135,000","Last sold\nAug 2010\n£152,995",Semi-detached house,2,1,1,Freehold,74 sqm,NaN,1.001346e+10,211000.0,222000.0,233000.0,53.772291,-1.164318,https://www.zoopla.co.uk/property/uprn/1001345...


# Data Cleaning

In [33]:
# The raw Data that was Scrapped from Zoopla will now be cleaned to make our data move readable

def Clean_data(df):
    # drop the duplicates from the data
    df = df.drop_duplicates()
    
    # Extract postcode using regex (UK postcodes pattern)
    df["Postcode"] = df["Address"].str.extract(r'([A-Z]{1,2}\d{1,2}[A-Z]?\s?\d[A-Z]{2})')
    
    # Extract town/city names 
    df["Post town name"] = df["Address"].str.extract(r',\s*([\w\s-]+),\s*[A-Z]{1,2}\d{1,2}[A-Z]?\s?\d[A-Z]{2}$')

    # Replace sqm with ""
    df["Square foot (sqm)"]=df["Square foot"].str.replace('sqm', '')
    
    # Extract Date and Amount from 'Last Sold'
    df[['Last Sold Date', 'Last Sold Amount']] = df['Date Last Sold'].str.extract(r"Last sold (\w+ \d{4})\n(£[\d,]+)")

    df['Last Sold Amount £']=df['Last Sold Amount'].str.replace('£', '')
    
    # Extract date from previous transaction(-1)
    df['previous date sold(-1)'] = df['Previous date sold(-1)'].str.extract(r'([A-Za-z]+\s\d{4})')
    
    # Extract the amount from previous transaction(-1)
    df['previous amount sold(-1)'] = df['Previous date sold(-1)'].str.extract(r'(£[\d,]+)')

    # replace £ from previous transaction (-1)
    df['previous amount sold(-1) £'] = df['previous amount sold(-1)'].str.replace('£', '')
    
    # Extract date from previous transaction(-2)
    df['previous date sold(-2)'] = df['Previous date sold(-2)'].str.extract(r'([A-Za-z]+\s\d{4})')
    
    # Extract the amount from previous transaction(-2)
    df['previous amount sold(-2)'] = df['Previous date sold(-2)'].str.extract(r'(£[\d,]+)')

    # replace £ from previous transaction (-2)
    df['previous amount sold(-2) £'] = df['previous amount sold(-2)'].str.replace('£', '')

    # Concatenate previous sale dates
    # df["Previous Dates sold"] = df["previous date sold(-1)"].astype(str) + " , " + df["previous date sold(-2)"].astype(str)

    # Concatenate previous amount
    # df["Previous Amount sold"] = df["previous amount sold(-1) £"].astype(str) + " " + df["previous date sold(-2) £"].astype(str)

In [35]:
Clean_data(df) # csll the function name

In [37]:
df

,Address,Date Last Sold,Previous date sold(-1),Previous date sold(-2),Property Type,Number of rooms,Number of Bath,Reception,Tenure,Square foot,...,Square foot (sqm),Last Sold Date,Last Sold Amount,Last Sold Amount £,previous date sold(-1),previous amount sold(-1),previous amount sold(-1) £,previous date sold(-2),previous amount sold(-2),previous amount sold(-2) £
0,"77 St George Building, 60 Great George Street,...","Last sold Dec 2024\n£129,950","Last sold\nSept 2013\n£110,000","Last sold\nAug 2006\n£144,000",Purpose built flat,2,1,,Leasehold,57 sqm,...,57,Dec 2024,"£129,950","129,950",Sept 2013,"£110,000","110,000",Aug 2006,"£144,000","144,000"
1,"Apartment 7, 49A St Pauls Street, Leeds, LS1 2FL","Last sold Dec 2024\n£204,500","Last sold\nSept 2019\n£198,000",,Purpose built flat,2,,,Leasehold,61 sqm,...,61,Dec 2024,"£204,500","204,500",Sept 2019,"£198,000","198,000",NaN,NaN,NaN
2,"32 St George Building, 60 Great George Street,...","Last sold Dec 2024\n£232,000","Last sold\nSept 2013\n£135,000","Last sold\nOct 2006\n£198,432",Flat/Maisonette,2,,,Leasehold,84 sqm,...,84,Dec 2024,"£232,000","232,000",Sept 2013,"£135,000","135,000",Oct 2006,"£198,432","198,432"
3,"Flat 146, Candle House, 1 Wharf Approach, Leed...","Last sold Dec 2024\n£175,000","Last sold\nMay 2017\n£168,950","Last sold\nMay 2014\n£139,950",Purpose built flat,1,1,1,Leasehold,50 sqm,...,50,Dec 2024,"£175,000","175,000",May 2017,"£168,950","168,950",May 2014,"£139,950","139,950"
4,"16 St George Building, 60 Great George Street,...","Last sold Oct 2024\n£140,000","Last sold\nSept 2006\n£129,200",,Flat/Maisonette,1,,,Leasehold,60 sqm,...,60,Oct 2024,"£140,000","140,000",Sept 2006,"£129,200","129,200",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24595,"10 Dawlish Mount, Leeds, LS9 9DZ","Last sold Jul 2022\n£110,000","Last sold\nAug 2013\n£45,000","Last sold\nJul 2006\n£88,000",Terrace property,3,1,1,Freehold,92 sqm,...,92,Jul 2022,"£110,000","110,000",Aug 2013,"£45,000","45,000",Jul 2006,"£88,000","88,000"
24596,"30 Skelton Road, Leeds, LS9 9EP","Last sold Jul 2022\n£140,000","Last sold\nOct 2018\n£110,000","Last sold\nSept 2014\n£79,995",Mid terrace house,2,1,1,Freehold,80 sqm,...,80,Jul 2022,"£140,000","140,000",Oct 2018,"£110,000","110,000",Sept 2014,"£79,995","79,995"
24597,"Apartment 203, Trinity One, East Street, Leeds...","Last sold Jul 2022\n£130,000","Last sold\nMay 2015\n£125,000","Last sold\nJun 2007\n£150,000",Flat/Maisonette,2,1,1,Leasehold,60 sqm,...,60,Jul 2022,"£130,000","130,000",May 2015,"£125,000","125,000",Jun 2007,"£150,000","150,000"
24598,"13 St Alban Crescent, Leeds, LS9 6JY","Last sold Jul 2022\n£137,000",,,Semi-detached house,3,1,1,Freehold,89 sqm,...,89,Jul 2022,"£137,000","137,000",NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
df.to_csv(r"C:\Users\obinn\OneDrive\Desktop\Data\House\Leeds\Leeds.csv", index=False)

In [7]:
df.head(5)

,Address,Date Last Sold,Previous date sold(-1),Previous date sold(-2),Property Type,Number of rooms,Number of Bath,Reception,Tenure,Square foot,EPC Rating,UPRN,lowerPrice,currentPrice,upperPrice,longitude,latitude,Listing URL
0,"Sunbridge Halls, 178 Sunbridge Road, Bradford,...","Last sold Jan 2025\n£13,500","Last sold\nOct 2022\n£10,000","Last sold\nSept 2022\n£10,500",Flat/Maisonette,1,1,1,Leasehold,,NaN,1.000519e+11,67000.0,84000.0,100000.0,53.796735,-1.764231,https://www.zoopla.co.uk/property/uprn/1000519...
1,"Flat 1, Sunbridge House, 80 Kirkgate, Bradford...","Last sold Dec 2024\n£55,000","Last sold\nDec 2006\n£148,400",,Flat/Maisonette,3,2,1,Leasehold,76 sqm,EPC rating: D,1.007006e+10,69000.0,87000.0,104000.0,53.794068,-1.756230,https://www.zoopla.co.uk/property/uprn/1007005...
2,"Flat 26, Equity Chambers, 40 Piccadilly, Bradf...","Last sold Dec 2024\n£65,000","Last sold\nSept 2006\n£142,000",,Converted flat,2,2,1,Leasehold,89 sqm,EPC rating: D,1.001058e+10,78000.0,97000.0,117000.0,53.796419,-1.754395,https://www.zoopla.co.uk/property/uprn/1001058...
3,"Apartment 42, City Exchange, 61 Hall Ings, Bra...","Last sold Dec 2024\n£159,000",,,,,,,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.zoopla.co.uk/property/uprn/1009486...
4,"Apartment 109, City Exchange, 61 Hall Ings, Br...","Last sold Nov 2024\n£182,000","Last sold\nDec 2018\n£75,000",,,,,,NaN,,NaN,1.009487e+10,59000.0,74000.0,89000.0,53.791212,-1.752619,https://www.zoopla.co.uk/property/uprn/1009486...


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
from datetime import datetime
import os

# === File paths ===
base_path = r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\2_integration_aggregation\2_house_price\3_outputs\1_house_onspd"

files_to_merge = [
    "1.0_exact.match_house_onspd.csv",
    "1.1_fuzzy.match_house_onspd.csv",
    "1.2_reverse.geo_match_house_onspd.csv",
    "1.3_fuzzy_reverse.geo.match_house_onspd.csv"
]
shapefile_path = os.path.join(base_path, "lsoa_e12000003.shp")
output_csv = os.path.join(base_path, "1.4_final_house_onspd.csv")
log_path = os.path.join(base_path, "1.4_final_house_onspd_log.txt")
density_map_output = os.path.join(base_path, "house_density_map.png")

# === Step 1: Merge rental data ===
dfs = [pd.read_csv(os.path.join(base_path, f)) for f in files_to_merge]
house_df = pd.concat(dfs, ignore_index=True).drop_duplicates(subset="new_id")

# === Step 2: Convert rental data to GeoDataFrame ===
house_gdf = gpd.GeoDataFrame(
    house_df,
    geometry=gpd.points_from_xy(house_df.longitude, house_df.latitude),
    crs="EPSG:4326"
)

# === Step 3: Load LSOA shapefile for E12000003 ===
lsoa_gdf = gpd.read_file(shapefile_path)
lsoa_gdf.columns = lsoa_gdf.columns.str.lower()
lsoa_gdf = lsoa_gdf.to_crs("EPSG:4326")  # Ensure coordinate systems match

# === Step 4: Filter house to E12000003 only ===
house_within = gpd.sjoin(house_gdf, lsoa_gdf, predicate="within", how="inner")

# === Step 5: Count house listings per LSOA ===
house_counts = house_within['lsoa21cd'].value_counts().rename("house_count")
lsoa_gdf = lsoa_gdf.merge(house_counts, left_on="lsoa21cd", right_index=True, how="left")
lsoa_gdf["house_count"] = lsoa_gdf["house_count"].fillna(0)

# === Step 6: Plot point density map ===
fig, ax = plt.subplots(1, 1, figsize=(10, 12))
lsoa_gdf.plot(column="house_count", cmap="OrRd", legend=True,
              legend_kwds={"label": "house Listings per LSOA"},
              edgecolor="black", linewidth=0.3, ax=ax)
ax.set_title("house Listing Density Across LSOAs in E12000003", fontsize=14)
ax.axis("off")
plt.savefig(density_map_output, dpi=300, bbox_inches="tight")

# === Step 7: Save output and log ===
house_within.drop(columns="index_right").to_csv(output_csv, index=False)

with open(log_path, "w") as f:
    f.write(f"Filtered house Density Log - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("-" * 60 + "\n")
    f.write(f"house records before filtering: {len(house_df)}\n")
    f.write(f"house records within E12000003: {len(house_within)}\n")
    f.write(f"Unique LSOAs with house: {lsoa_gdf[lsoa_gdf['house_count'] > 0].shape[0]}\n")
    f.write(f"Density map saved to: {density_map_output}\n")


In [2]:
# -*- coding: utf-8 -*-
# City master
#Now that each city folder contains a "*_combined.csv" file, we merge them all into a single master city dataset, containing housing prices from Zoopla,
# over the period February-June 2025.

import os
import pandas as pd
from datetime import datetime
import re

# Base directory with city folders
base_dir = r"C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\1_raw_data\1.1_house_price\1_raw_data"

# Output paths
master_path = os.path.join(base_dir, "0.1_raw_house_city_master.csv")
log_path = os.path.join(base_dir, "0.1_raw_house_city_master_log.txt")

# List all subdirectories
cities = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]


# ===== Step 1: Merge all *_combined.csv files =====
combined_files = []
found_files = 0
missing_files = []

for city in cities:
    combined_path = os.path.join(base_dir, city, f"{city}_combined.csv")
    if os.path.exists(combined_path):
        print(f"Found: {combined_path}")
        df = pd.read_csv(combined_path)
        combined_files.append(df)
        found_files += 1
    else:
        print(f"Missing: {combined_path}")
        missing_files.append(city)


# Merge into master DataFrame
master_df = pd.concat(combined_files, ignore_index=True)
initial_rows = len(master_df)


# ===== Step 2: Keep only rows where Amount looks like rent =====

# def is_clean_rent(value):
#     try:
#         # Remove commas, normalize encoding
#         value = str(value).replace(",", "").strip()
#         value = value.encode('utf-8', errors='ignore').decode('utf-8', errors='ignore')

#         # Convert
#         value = value.replace("\u00C2\u00A3", "\u00A3")

#         # Match:
#         return bool(re.match(r"^\u00A3\d{2,5}(\.\d{1,2})?\s*(pcm|per month)?$", value, re.IGNORECASE))
#     except:
#         return False

# # Apply filter
# master_df["Amount"] = master_df["Amount"].astype(str).str.strip()
# valid_mask = master_df["Amount"].apply(is_clean_rent)
# rows_removed = (~valid_mask).sum()
# master_df = master_df[valid_mask]
# final_rows = len(master_df)


# ===== Step 3: Save cleaned master file =====
master_df.to_csv(master_path, index=False)
print(f"Master file saved to: {master_path}")

# ===== Step 4: Write summary log =====
with open(log_path, "w") as log:
    log.write(f"Merge Log - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    log.write(f"---------------------------------------------\n")
    log.write(f"Total city folders scanned: {len(cities)}\n")
    log.write(f"Files successfully merged: {found_files}\n")
    # log.write(f"Missing city folders: {', '.join(missing_files) if missing_files else 'None'}\n")
    # log.write(f"Initial total rows before filtering: {initial_rows}\n")
    # log.write(f"Rows removed because Amount did not start with \\u00C2\\u00A3 or \\u00A3 and digits: {rows_removed}\n")
    # log.write(f"Final cleaned rows: {final_rows}\n")
    log.write(f"Saved master dataset to: {master_path}\n")

print(f"Log file saved to: {log_path}")

Found: C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\1_raw_data\1.1_house_price\1_raw_data\Bradford\Bradford_combined.csv
Found: C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\1_raw_data\1.1_house_price\1_raw_data\Calderdale\Calderdale_combined.csv
Found: C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\1_raw_data\1.1_house_price\1_raw_data\Darlington\Darlington_combined.csv
Found: C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\1_raw_data\1.1_house_price\1_raw_data\Doncaster\Doncaster_combined.csv
Found: C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\1_raw_data\1.1_house_price\1_raw_data\Harrogate\Harrogate_combined.csv
Found: C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\1_raw_data\1.1_house_price\1_raw_data\Hull\Hull_combined.csv
Found: C:\Users\vbvb850\OneDrive - University of Leeds\Desktop\Data\1_raw_data\1.1_house_price\1_raw_data\Kirklees\Kirklees_combined.csv
Found: C:\Users\vbvb850\OneDrive - Un